# Credit Scorecard Development
## Notebook 1 — Exploratory Data Analysis

This notebook explores the credit application dataset to understand:
- Portfolio composition and demographics
- Missing value patterns
- Feature distributions vs default status
- Correlation structure

**Dataset:** Synthetic credit application data (n = 15,000) simulating Thai commercial bank lending.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import load_credit_data

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = load_credit_data()
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Basic Statistics & Missing Values

In [ ]:
print('=== Data Types & Non-null Counts ===')
print(df.info())

print('\n=== Descriptive Statistics ===')
df.describe().round(2)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]

print('=== Missing Value Summary ===')
print(missing_df)

fig, ax = plt.subplots(figsize=(8, 4))
missing_df['missing_pct'].plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.8)
ax.set_xlabel('Missing Rate (%)')
ax.set_title('Missing Values by Feature')
plt.tight_layout()
plt.savefig('../plots/01_missing_values.png', bbox_inches='tight')
plt.show()

## 2. Default Rate Overview

In [ ]:
default_rate = df['default'].mean()
print(f'Overall Default Rate: {default_rate:.2%}')
print(f'Defaults: {df["default"].sum():,}  |  Non-defaults: {(1-df["default"]).sum():,}')

fig, ax = plt.subplots(figsize=(5, 4))
labels = ['Non-Default', 'Default']
sizes = [1 - default_rate, default_rate]
colors = ['#2980b9', '#e74c3c']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Portfolio Default Rate')
plt.tight_layout()
plt.savefig('../plots/01_default_rate.png', bbox_inches='tight')
plt.show()

## 3. Feature Distributions by Default Status

In [ ]:
features = ['age', 'annual_income', 'employment_years', 'loan_amount',
            'credit_utilization', 'num_delinquencies', 'debt_to_income']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    for label, color in [(0, '#2980b9'), (1, '#e74c3c')]:
        subset = df[df['default'] == label][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label='Non-Default' if label == 0 else 'Default', density=True)
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions by Default Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Default Rate by Feature Decile

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

cont_features = ['age', 'annual_income', 'employment_years',
                 'credit_utilization', 'debt_to_income', 'loan_amount']

for i, feat in enumerate(cont_features):
    tmp = df[[feat, 'default']].dropna()
    tmp['decile'] = pd.qcut(tmp[feat], q=10, duplicates='drop')
    dr = tmp.groupby('decile', observed=True)['default'].mean() * 100
    dr.plot(kind='bar', ax=axes[i], color='#e67e22', alpha=0.85, edgecolor='white')
    axes[i].set_title(f'Default Rate by {feat.replace("_"," ").title()} Decile')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Univariate Default Rate Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_default_rate_by_decile.png', bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = df[features + ['default']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/01_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('\n=== Correlation with Default (sorted) ===')
print(corr['default'].drop('default').sort_values(key=abs, ascending=False).round(4))